# 工作区、暂存区与提交

预计阅读时间：10-15 分钟。

Git 提交前要先理解三个位置：工作区、暂存区和当前提交。

## 原理

### 三个位置

| 位置 | 说明 |
| --- | --- |
| 工作区 | 正在编辑的文件 |
| 暂存区 | 准备进入下一次提交的文件 |
| 当前提交 | 当前分支指向的那次提交，通常也就是 `HEAD` 对应的提交 |

### 四种状态

常见文件状态包括：

- 未跟踪（untracked）：Git 还没有管理这个文件；
- 已修改（modified）：文件已经被 Git 管理，但工作区内容发生了变化；
- 已暂存（staged）：修改已经放入暂存区，准备进入下一次提交；
- 已提交（committed）：内容已经保存为一次提交，成为 Git 历史的一部分。

暂存区的作用，是让我们先挑选“哪些修改进入下一次提交”。这样一次提交可以只包含一个清晰变化，而不是把工作区里所有改动都混在一起。

## 原子操作

Git 的很多命令，本质上是在改变文件状态。先把常见状态转换拆开看。

### 从未跟踪到已暂存

新建文件后，文件处于 `untracked` 状态。使用 `git add` 可以把它加入暂存区：



In [ ]:
git add notes.md




```mermaid
flowchart LR
    U[untracked\n未跟踪] -->|git add| S[staged\n已暂存]
    classDef work fill:#f8fafc,stroke:#64748b,color:#0f172a;
    classDef staged fill:#ecfdf5,stroke:#059669,color:#064e3b;
    class U work;
    class S staged;
```

### 从已修改到已暂存

已经被 Git 跟踪的文件，修改后会变成 `modified`。再次使用 `git add`，可以把这次修改加入暂存区：



In [ ]:
git add app.py




```mermaid
flowchart LR
    M[modified\n已修改] -->|git add| S[staged\n已暂存]
    classDef work fill:#f8fafc,stroke:#64748b,color:#0f172a;
    classDef staged fill:#ecfdf5,stroke:#059669,color:#064e3b;
    class M work;
    class S staged;
```

### 从已暂存到已提交

`git commit` 会把暂存区中的内容保存为一次提交：



In [ ]:
git commit -m "Add training script"




```mermaid
flowchart LR
    S[staged\n已暂存] -->|git commit| C[committed\n已提交]
    classDef staged fill:#ecfdf5,stroke:#059669,color:#064e3b;
    classDef committed fill:#eef2ff,stroke:#4f46e5,color:#111827;
    class S staged;
    class C committed;
```

### 取消暂存

如果文件已经 `git add`，但暂时不想提交，可以取消暂存：



In [ ]:
git restore --staged app.py




`git restore --staged app.py` 会让暂存区回到当前提交的版本，但不会修改工作区文件。

例如执行前：

```text
当前提交: 旧版本
暂存区:   新版本1
工作区:   新版本2
```

执行后：

```text
当前提交: 旧版本
暂存区:   旧版本
工作区:   新版本2
```

因此，文件通常会从 `staged` 回到 `modified` 状态：

```mermaid
flowchart LR
    S[staged\n修改已进入暂存区] -->|git restore --staged| M[modified\n工作区修改仍保留]
    classDef staged fill:#ecfdf5,stroke:#059669,color:#064e3b;
    classDef work fill:#f8fafc,stroke:#64748b,color:#0f172a;
    class S staged;
    class M work;
```

### 丢弃工作区修改

如果确定不要工作区中的当前修改，可以丢弃它：



In [ ]:
git restore app.py




`git restore app.py` 会让工作区文件回到暂存区中的版本；如果暂存区没有该文件的修改，就会回到当前提交的版本。

例如执行前：

```text
当前提交: 旧版本
暂存区:   新版本1
工作区:   新版本2
```

执行后：

```text
当前提交: 旧版本
暂存区:   新版本1
工作区:   新版本1
```

因此，`git restore app.py` 丢弃的是工作区里尚未暂存的修改：

```mermaid
flowchart LR
    W[working\n工作区新版本2] -->|git restore| S[staged\n暂存区新版本1]
    classDef work fill:#f8fafc,stroke:#64748b,color:#0f172a;
    classDef staged fill:#ecfdf5,stroke:#059669,color:#064e3b;
    class W work;
    class S staged;
```

这个命令会丢弃工作区修改，执行前要确认这些修改不再需要。

## 常见工作流

把上面的原子操作组合起来，就形成了日常使用 Git 时最常见的工作流。


### 新增文件的状态变化

新增文件时，可以把 `git add` 和 `git commit` 理解成文件状态在三个位置之间变化：

```mermaid
flowchart LR
    subgraph W[工作区]
        U[untracked\n新文件未跟踪]
    end

    subgraph S[暂存区]
        A[staged\n准备提交]
    end

    subgraph H[当前提交]
        C[committed\n已经提交]
    end

    U -- git add --> A
    A -- git commit --> C
```

`git add` 不是提交，它只是把文件加入下一次提交的候选列表。真正写入 Git 历史的是 `git commit`。

### 修改文件的状态变化

如果文件已经被提交过，再次修改时，状态变化通常是：

```mermaid
flowchart LR
    subgraph H1[当前提交]
        C1[committed\n已有提交]
    end

    subgraph W[工作区]
        M[modified\n文件已修改]
    end

    subgraph S[暂存区]
        A[staged\n准备提交]
    end

    subgraph H2[当前提交]
        C2[committed\n新的提交]
    end

    C1 -- 编辑文件 --> M
    M -- git add --> A
    A -- git commit --> C2
```

对于已经跟踪的文件，修改后会进入 `modified` 状态。再次运行 `git add` 后，修改内容才会进入暂存区。

## 示例

继续使用上一节的 `git-principle-demo`。如果当前不在该目录，先进入项目目录：



In [ ]:
cd git-principle-demo
git switch main




修改一个已跟踪文件，再新增一个文件：



In [ ]:
echo 'print("status demo")' >> app.py
echo 'temporary note' > notes.md
git status




此时，`app.py` 通常是 `modified`，`notes.md` 通常是 `untracked`。

把两个文件加入暂存区，并保存为一次提交：



In [ ]:
git add app.py notes.md
git status
git commit -m "Demonstrate file status"




`git add` 把指定文件的当前内容放入暂存区。`git commit` 把暂存区内容保存为一次提交。


## 提交粒度

上面的两个文件都服务于同一个状态演示，所以可以放在一次提交中。一般来说，一次提交应对应一个清晰变化，例如：

- 修改 `app.py` 的输出；
- 新增 `notes.md`；
- 修复一个具体 bug；
- 更新 `README.md`。

不要把很多无关修改混在一次提交里。

## 练习

请在一个独立目录中练习，不要在本课程项目仓库中直接操作，避免影响课程文件的 Git 状态。

创建一个新的练习仓库：



In [ ]:
mkdir git-status-practice
cd git-status-practice
git init -b main




创建文件、查看状态，并完成一次提交：



In [ ]:
echo 'print("hello")' > app.py
git status
git add app.py
git status
git commit -m "Add app"




继续修改 `app.py`，观察 `modified`、`staged` 和 `committed` 之间的变化。